# Topic 2 - categories / POS

Lexfile (coarse semantic class) and part of speech. See
[`05.54_data_enrich.md`](../../scratch_space/09_concept_model/05.54_data_enrich/05.54_data_enrich.md) Topic 2.

Open questions: lexfile coverage and distinct set; is lexfile concept-level
(on the ILI/English synset) or per language; the POS distribution and the
odd codes.

## Setup

Thin caller over the staged cache and the OMW wordnets.

In [ ]:
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import wn
from loguru import logger as lg

from lang_tools.lexicon.ingestion.sources.omw import OMW_LEXICONS, OMW_VERSION
from lang_tools.params.lang_tools_params import get_lang_tools_params

LANGS = ["en", "pt", "es", "fr", "it"]
paths = get_lang_tools_params().paths
data_fol = paths.data_fol
staging = data_fol / "_raw/lexicon/staging"
wn.config.data_directory = str(data_fol / "_raw/lexicon/wn_data")


def staged(dataset: str, lang: str) -> pd.DataFrame:
    """Read a staged parquet (``<staging>/<dataset>/<lang>.parquet``)."""
    return pq.read_table(staging / dataset / f"{lang}.parquet").to_pandas()


def wordnet(lang: str) -> wn.Wordnet:
    """Open the pinned OMW lexicon for a language (one wordnet, no merging)."""
    return wn.Wordnet(lexicon=f"{OMW_LEXICONS[lang]}:{OMW_VERSION}")


def ili_of(synset: wn.Synset) -> str | None:
    """Return the synset's ILI id as a plain string, or ``None``."""
    il = synset.ili
    return getattr(il, "id", il) or None


lg.info("staging at {}", staging)

## Lexfile + ILI coverage

In [ ]:
# Lexfile + ILI coverage per language, and the distinct lexfile set (en).
rows = []
en_lexfiles = Counter()
for lang in LANGS:
    n = n_lex = n_ili = 0
    for s in wordnet(lang).synsets():
        n += 1
        if s.lexfile():
            n_lex += 1
            if lang == "en":
                en_lexfiles[s.lexfile()] += 1
        if ili_of(s):
            n_ili += 1
    rows.append(
        {
            "lang": lang,
            "synsets": n,
            "with_lexfile": n_lex,
            "with_ili": n_ili,
            "ili_pct": round(100 * n_ili / n, 1),
        }
    )
lexfile_cov = pd.DataFrame(rows).set_index("lang")
print("distinct en lexfiles:", len(en_lexfiles))
lexfile_cov

## POS distribution

In [ ]:
# POS distribution per language (watch for the odd codes flagged in 05.4).
pos = {lang: Counter(s.pos for s in wordnet(lang).synsets()) for lang in LANGS}
pos_df = pd.DataFrame(pos).fillna(0).astype(int)
pos_df

## Findings (measured 2026-06-21)

- **lexfile lives on the English synset only** (en 100%, all others 0%), but
  every synset in every language is ILI-linked (100%), so lexfile resolves to
  the concept through the shared ILI. It is concept-level with an English
  source, exactly like examples. 45 distinct lexfiles (the standard WordNet
  set, e.g. `noun.motion`).
- **POS is clean at the synset level:** only `n v a s r` appear; `s` is the
  satellite adjective already mapped to `adjective` in `sources.omw`. None of
  the odd `p / x / u` codes from the 05.4 checks show up on synsets (those were
  lemma-level), so synset POS needs no extra cleanup here.

**Decision for Step 4:** promote lexfile as a concept-level field keyed on ILI
(sourced from the English synset), feeding the phase-6 clustering. POS keeps the
existing `_POS_LABELS` mapping; no new cleanup scope from synset POS.